# holdout-data-one-per-class — faded example 3: Faded: seeded random per-class holdout using local Generator

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `holdout-data-one-per-class`. The last cell reports your progress on the `Generative: Hold-out one-per-class data` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Hold-out one-per-class data` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`holdout-data-one-per-class`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "holdout-data-one-per-class"
DD_SUBTOPIC = "Generative: Hold-out one-per-class data"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For a reproducible but randomly chosen holdout, create a local `t.Generator` seeded with a fixed value. For each class, use `(labels == c).nonzero(as_tuple=True)[0]` to get the indices of that class in the dataset, then `t.randint(len(idxs), (1,), generator=g).item()` to randomly pick one index into that list. A local generator isolates the pick from any global `torch.manual_seed` calls in surrounding code.

## Faded exercise 3

Complete `seeded_one_per_class`. The blank is the index-sampling step: given `idxs` (the positions in `labels` where `labels == c`), randomly pick one using the local generator `g`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(0)

def seeded_one_per_class(data: t.Tensor, labels: t.Tensor, num_classes: int, seed: int) -> t.Tensor:
    g = t.Generator().manual_seed(seed)
    per_class = []
    for c in range(num_classes):
        idxs = (labels == c).nonzero(as_tuple=True)[0]
        pick = t.randint(len(idxs), (1,), generator=g).item()
        per_class.append(data[idxs[pick].item()])
    return t.stack(per_class, dim=0)

# Exercise it
t.manual_seed(0)
N, num_classes, D = 40, 4, 5
data = t.randn(N, D)
base = t.arange(N) % num_classes
perm = t.randperm(N)
data, labels = data[perm], base[perm]
h1 = seeded_one_per_class(data, labels, num_classes, seed=42)
h2 = seeded_one_per_class(data, labels, num_classes, seed=42)
print(t.allclose(h1, h2))  # True: same seed -> same gallery


def _test():
    import torch as t
    t.manual_seed(0)
    N, C, D = 50, 5, 6
    data = t.randn(N, D)
    base = t.arange(N) % C
    perm = t.randperm(N)
    data, labels = data[perm], base[perm]

    # Same seed -> identical galleries
    h1 = seeded_one_per_class(data, labels, C, seed=7)
    h2 = seeded_one_per_class(data, labels, C, seed=7)
    assert t.allclose(h1, h2), "Same seed must produce identical galleries"
    assert h1.shape == (C, D), f"Shape {h1.shape} != ({C}, {D})"

    # Different seed -> usually different (probabilistic but almost certain for large N)
    h3 = seeded_one_per_class(data, labels, C, seed=999)
    # verify all selected samples belong to the right class
    for c in range(C):
        class_data = data[labels == c]
        assert any(t.allclose(h1[c], row) for row in class_data), \
            f"Class {c} holdout not found among class-{c} samples"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

def seeded_one_per_class(data: t.Tensor, labels: t.Tensor, num_classes: int, seed: int) -> t.Tensor:
    g = t.Generator().manual_seed(seed)
    per_class = []
    for c in range(num_classes):
        idxs = (labels == c).nonzero(as_tuple=True)[0]
        pick = t.randint(len(idxs), (1,), generator=g).item()
        per_class.append(data[idxs[pick].item()])
    return t.stack(per_class, dim=0)

# Exercise it
t.manual_seed(0)
N, num_classes, D = 40, 4, 5
data = t.randn(N, D)
base = t.arange(N) % num_classes
perm = t.randperm(N)
data, labels = data[perm], base[perm]
h1 = seeded_one_per_class(data, labels, num_classes, seed=42)
h2 = seeded_one_per_class(data, labels, num_classes, seed=42)
print(t.allclose(h1, h2))  # True: same seed -> same gallery
```
</details>